# StyleGraph curated catalog batch processor

This notebook runs offline catalog processing. It keeps raw images and writes processed PNGs, manifests, and QA reports locally in the runtime.


In [ ]:
!pip install -q pillow rembg onnxruntime google-genai pydantic


In [ ]:
import os, sys, zipfile
from pathlib import Path
sys.path.insert(0, '/content/autofashion')
from curated_pipeline.batch_processor import BatchProcessor, GeminiCuratedMetadataProvider, export_processed_zip
from curated_pipeline.cache import DatasetLayout
FORCE_REPROCESS = False
WORK_ROOT = Path('/content/stylegraph_curated')
DATASET_ROOT = WORK_ROOT / 'dataset'
GEMINI_MODEL = os.getenv('GEMINI_CURATED_MODEL', 'gemini-2.5-flash')
print('Model:', GEMINI_MODEL)


## Input
Upload a ZIP containing `raw/feminine`, `raw/masculine`, or `raw/neutral` folders, or copy a Drive folder into the runtime. Keep the category folder names (`tops`, `bottoms`, `one_piece`, `outerwear`, `footwear`, `accessories`) so hints are preserved.


In [ ]:
# ZIP option
from google.colab import files
uploaded = files.upload()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for name, data in uploaded.items():
    archive = WORK_ROOT / name
    archive.write_bytes(data)
    with zipfile.ZipFile(archive) as z: z.extractall(DATASET_ROOT)


In [ ]:
# Drive option (run instead of the ZIP cell when using Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_ROOT = Path('/content/drive/MyDrive/stylegraph_curated_dataset')


In [ ]:
from rembg import new_session, remove
session = new_session('isnet-general-use')
def remove_background(data, mime):
    return remove(data, session=session)
provider = GeminiCuratedMetadataProvider(api_key=os.environ['GEMINI_API_KEY'], model=GEMINI_MODEL)
layout = DatasetLayout(DATASET_ROOT)
processor = BatchProcessor(layout, provider, background_remover=remove_background, force_reprocess=FORCE_REPROCESS)
summary, records, failures = processor.run()
summary.as_dict()


In [ ]:
# QA gallery: up to twelve safe records (no image bytes or credentials are displayed)
from PIL import Image, ImageDraw
for record in records[:12]:
    print(record.get('display_name'), record.get('category'), record.get('subcategory'), record.get('model_confidence'), record['status'])


In [ ]:
export_path = export_processed_zip(DATASET_ROOT, WORK_ROOT / 'stylegraph_curated_processed.zip')
print(f'Exported: {export_path}')
print('Download this ZIP, extract it locally, review reports/results.json and reports/failed.json, then run the existing local upload command against the extracted processed/manifests folders.')
